# Dealing with Mysterious Conflicts

This notebook is a variant of `05-Mysterious-Conflicts.ipynb` that uses `Larkita` instead of `Lark`.  It discusses
*reduce/reduce* conflicts that result from the fact that the set of `LR` states is compressed into the set of `LALR`
states.  Such a conflict is called *mysterious*, because it is not caused
by any defect of the grammar:  the grammar below is unambiguous and it even has the `LR(1)` property.  The
conflict appears only when two `LR` states that have the same *core* are merged into a single `LALR` state.

We discuss the following grammar:
```
    s : "v" a "y"
      | "w" b "y"
      | "v" b "z"
      | "w" a "z"

    a : X     -> a

    b : X     -> b

    X : "x"
```
The language generated by this grammar consists of the four strings `vxy`, `vxz`, `wxy` and `wxz`.  Whether
the `x` in the middle is an `a` or a `b` is determined by the *combination* of the first and the last
character:  in `vxy` and `wxz` it is an `a`, while in `wxy` and `vxz` it is a `b`.

`Lark` generates `LALR` parsers and therefore runs into these conflicts.  `Larkita` always generates *canonical*
`LR(1)` parsers, which never merge states.  We will see that `Larkita` has no problem with this grammar.

## Specification of the Grammar

The token classes and the grammar rules are given in a single grammar.  The characters `v`, `w`, `y` and `z`
occur as *string literals* inside the rules.  As they form the heads of the abstract syntax trees, a string like
`vxy` yields a tuple with the head `('v', 'y')`.  The *aliases* `-> a` and `-> b` give the abstract syntax trees of the
rules `a` and `b` different heads.  Without them, both rules would pass the token `X` through and we could not see
which of the two rules has been used.

In [ ]:
grammar = r"""
    s : "v" a "y"
      | "w" b "y"
      | "v" b "z"
      | "w" a "z"

    a : X     -> a

    b : X     -> b

    X : "x"

    %import common.WS
    %ignore WS
"""

## Building the `LR(1)` Parser

If the LR(1) automaton has a *reduce/reduce* conflict, `Larkita` raises an exception of class `GrammarError`.
For shift/reduce conflicts, it issues a `ConflictWarning`.  We turn this warning into an exception, too.

In [ ]:
import warnings

from larkita import Larkita, ConflictWarning, GrammarError

In [ ]:
with warnings.catch_warnings():
    warnings.simplefilter('error', ConflictWarning)
    try:
        parser = Larkita(grammar, out_file='mysterious.out')
        print('No conflict.')
    except (GrammarError, ConflictWarning) as e:
        print(e)

## Why Is There No Conflict?

If we compute the `LR(1)` states of this grammar, we find the two states
$$ \{\; a \rightarrow X \bullet : \texttt{'y'}, \quad b \rightarrow X \bullet : \texttt{'z'} \;\}
   \qquad\text{and}\qquad
   \{\; b \rightarrow X \bullet : \texttt{'y'}, \quad a \rightarrow X \bullet : \texttt{'z'} \;\} $$
The first of these states is reached after reading `vx`, the second one after reading `wx`.

Taken by itself, *neither* of these two states has a conflict, since the follow sets of the two rules are
disjoint in both cases.  This is precisely the reason why the grammar has the `LR(1)` property.

However, the two states have the same *core*
$$ \{\; a \rightarrow X \bullet, \quad b \rightarrow X \bullet \;\} $$
and therefore they would be merged when an `LALR` table is computed.  The resulting state would be
$$ \{\; a \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\}, \quad
       b \rightarrow X \bullet : \{\texttt{'y'}, \texttt{'z'}\} \;\} $$
and this state obviously has a *reduce/reduce* conflict, both for the token `'y'` and for the token `'z'`.
The merger throws away the information about which of the two states we came from, and it is precisely
this information that is needed to decide between `a` and `b`.  The grammar is therefore an `LR(1)` grammar, but
*not* an `LALR(1)` grammar.

The canonical `LR(1)` automaton computed by `Larkita` keeps the two states apart.  The following cell prints the two
states from the file `mysterious.out`:  Each of them reduces a `'y'` with one rule and a `'z'` with the other.

In [ ]:
with open('mysterious.out') as handle:
    report = handle.read()
for state in report.split('\n\n\nState ')[1:]:
    if 'a → X •' in state:
        print('State ' + state + '\n')

## Parsing the Strings of the Language

All four strings of the language are parsed correctly.

In [ ]:
for s in ['vxy', 'vxz', 'wxy', 'wxz']:
    print(s, parser.parse(s))

Observe that the middle `x` is recognized as an `a` in `vxy` and `wxz`, but as a `b` in `vxz` and `wxy`.